In [9]:
import pandas as pd
from prophet import Prophet
import joblib

In [11]:
# 1. Load the dataset
df = pd.read_csv('wfp_food_prices_zwe.csv')

In [12]:
# 2. Pre-processing
# Prophet requires 'ds' (date) and 'y' (target)
df['ds'] = pd.to_datetime(df['date'])
df['y'] = df['usdprice']
df['admin1'] = df['admin1'].fillna('Unknown')

# We need to aggregate to avoid duplicate dates for the same commodity/region
df_agg = df.groupby(['ds', 'commodity', 'admin1'])['y'].mean().reset_index()

In [13]:
# 3. Handle Categorical Data (One-Hot Encoding)
# This allows one model to understand many commodities and regions
df_prophet = pd.get_dummies(df_agg, columns=['commodity', 'admin1'])
commodity_cols = [col for col in df_prophet.columns if col.startswith('commodity_') or col.startswith('admin1_')]

In [14]:
# 4. Initialize and Train the Global Model
model = Prophet(yearly_seasonality=True, daily_seasonality=False)

# Add each commodity as a 'regressor'
for col in commodity_cols:
    model.add_regressor(col)

model.fit(df_prophet)

17:16:05 - cmdstanpy - INFO - Chain [1] start processing
17:16:06 - cmdstanpy - INFO - Chain [1] done processing


In [15]:
# 5. Save the Model and the Column List
# We save the columns so the API knows the exact order of the "switches"
joblib.dump(model, 'demand_forecast_model.pkl')
joblib.dump(commodity_cols, 'forecast_features.pkl')

print("Model and features saved successfully!")

Model and features saved successfully!
